# Orchestrator fine-tuning · Laya multilingual (whole model, RLCD) on Kaggle

Plan: `orchestrator/fine-tuning/plans/FT-P1-laya.md` (B0 notes: `FT-P1-laya-B0.md`). Stages stop at the first failure:
`b0` (read upstream, data check) → `b1` smoke → `b2` pilot (time projection) → `b3` full run (4 epochs, calibration) → `b4_validation` (report) → `b4_test` (**only when `FROZEN = True`**, once).

**Before you run**
- Accelerator: **one GPU T4**. Internet: **On**. Add Input: the private Dataset built by `fine-tuning/laya/make_bundle_laya.py`.
- Optional secret `HF_TOKEN`. Zero money: free tier only (FT-C-08). Never run two training sessions on one account. Single process, no DDP.
- Run `b0` to `b2` first and read the projection; then `b3` and `b4_validation`. Set `FROZEN = True` by hand only after validation is reviewed (FT-C-14).
- The checkpoint (about 650 MB) is saved in `/kaggle/working/runs/laya/<RUN_ID>/checkpoint`: download it, upload it as a private Kaggle Dataset, record the sha256 printed at the end and the Dataset name (pass it to `run_ft_laya.py --kaggle-dataset`).

In [ ]:
# ---- CONFIG (edit here) ----
STAGES = ["b0", "b1"]               # then add "b2", then "b3", "b4_validation"; "b4_test" needs FROZEN = True
FROZEN = False                      # set True by hand after validation is reviewed (FT-C-14): test is run once
RUN_ID = "laya-ft-multilingual-a"   # FT-P-03; a second configuration takes suffix -b
KAGGLE_DATASET = None               # name of the private Dataset holding the checkpoint, once uploaded (FT-P-06)
SESSION_HOURS = 12                  # Kaggle GPU session length: RE-CHECK on the account page (chapter 09 item 6), note it in run.json
SANITY = 0.395                      # FT-C-12: v2 zero-shot route macro-F1 (laya-v2-a); train_laya.py applies it on the development split

import os
os.environ["USE_TF"] = "0"          # chapter 09: laya imports can hang with TensorFlow installed
OUT = "/kaggle/working/runs/laya"
os.environ["HF_HOME"] = "/tmp/hf"
print(RUN_ID, STAGES, "FROZEN =", FROZEN)

In [ ]:
import subprocess, sys, shlex, glob, json, shutil

def sh(cmd, cwd=None, check=True, env=None):
    """Run a command, stream its output into the cell, raise on failure."""
    print("$", cmd if isinstance(cmd, str) else shlex.join(cmd), flush=True)
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")
    return p.returncode

sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv")
sh("python --version; df -h /tmp /kaggle/working | cat; free -g | head -2")

try:
    from kaggle_secrets import UserSecretsClient
    try: os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: pass
except ImportError:
    pass   # not on Kaggle

hits = glob.glob("/kaggle/input/**/orchestrator/fine-tuning/laya/train_laya.py", recursive=True)
assert len(hits) == 1, f"attach exactly one Dataset built by laya/make_bundle_laya.py; found {hits}"
FT = os.path.dirname(os.path.dirname(hits[0]))             # .../orchestrator/fine-tuning (read-only)
BASE = os.path.join(os.path.dirname(FT), "baselines")
BUNDLE = os.path.dirname(os.path.dirname(FT))
assert os.path.isfile(f"{BASE}/common/data.py") and os.path.isfile(f"{FT}/data/laya/train.jsonl") and os.path.isfile(f"{FT}/data/manifest.json")
assert os.path.isdir(f"{BUNDLE}/data-generation/data/final_v2/test"), "v2 data folder missing from the Dataset: rebuild the bundle"
print("bundle:", FT)
RUN = f"{OUT}/{RUN_ID}"

In [ ]:
# ---- environment + GPU check + tests ----
sh(f"pip install -q -r {BASE}/laya/requirements-laya.txt")
sh([sys.executable, "-c", """
import torch, laya
assert torch.cuda.is_available(), "no CUDA GPU: set Accelerator to a T4"
print("laya", laya.__version__, "| torch", torch.__version__, "| device", torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())
assert laya.__version__ == "0.4.31", "round-1 version is 0.4.31"
"""])
sh(f"{sys.executable} -m pytest -q -p no:cacheprovider common/tests laya/tests", cwd=FT, env={"PYTHONPATH": BUNDLE})
sh(f"{sys.executable} -c \"import sys; sys.path.append('common'); import checks; checks.check_manifest('data', prefix='laya/'); print('data manifest ok (laya files)')\"", cwd=FT)
# Kaggle quota: read the current session length and weekly GPU limit from the account page and write them into the notes (chapter 09 item 6).

In [ ]:
# ---- b0: read upstream at the installed version (chapter 09 item 4, FT-P1-laya-B0.md) ----
if "b0" in STAGES:
    sh([sys.executable, "-c", """
import inspect, json, os, laya, laya.train as LT, laya.agent as LA
from huggingface_hub import HfApi, snapshot_download
print("train_model", inspect.signature(LT.train_model))
print("TrainConfig defaults:", {k: v for k, v in LT.TrainConfig().__dict__.items() if k in ("epochs","micro_batch","grad_accum","encoder_lr","head_lr","min_lr","loss","shuffle_options","label_smoothing")})
src = inspect.getsource(LA.Agent._decode_answers)
assert "raw_rows[j] / t_scale" in src, "temperature form changed: stop and re-read open item 2"
print("temperature form: z = raw_rows[j] / t_scale  (raw = softmax(T * log p_cal))")
from laya.calibrate import MIN_BUCKET_N, MIN_TYPE_N
print("MIN_BUCKET_N", MIN_BUCKET_N, "MIN_TYPE_N", MIN_TYPE_N)
sha = HfApi().model_info("convaiinnovations/laya").sha
d = snapshot_download("convaiinnovations/laya", revision=sha, allow_patterns=["multilingual/rl_agent_config.json"])
cfg = json.load(open(os.path.join(d, "multilingual", "rl_agent_config.json")))
print("revision", sha, "| max_len", cfg["max_len"], "| head_max_len", cfg["head_max_len"], "| temperature", cfg["temperature"], "| by_options", cfg["temperature_by_options"])
assert (cfg["max_len"], cfg["head_max_len"]) == (1024, 256)
"""])
    print("B0 done: copy the printed lines into run.json notes (chapter 09 item 4).")

In [ ]:
# ---- b1: smoke. 64 train rows, 1 epoch, 40 development rows; reload the checkpoint with laya.load and run 4 rows through the harness ----
if "b1" in STAGES:
    SM = f"{OUT}/{RUN_ID}-b1"
    shutil.rmtree(SM, ignore_errors=True)                  # scratch run: start clean
    sh(f"{sys.executable} laya/train_laya.py --data-dir data/laya --out {SM} --limit 64 --dev-limit 40 --epochs 1 --no-sanity", cwd=FT)
    rep = json.load(open(f"{SM}/train_report.json"))
    assert rep["skipped"] == {"train": {}, "calibration": {}, "development": {}}, rep["skipped"]
    assert all(h["mean_loss"] is None or h["mean_loss"] == h["mean_loss"] for h in rep["epochs"]) and all(l == l and abs(l) < 1e6 for l in rep["epoch_mean_loss"]), "loss not finite"
    assert (rep["max_len"], rep["head_max_len"]) == (1024, 256)
    print("temperatures", rep["temperature"], "| clamped", rep["temperature_clamped"])
    sh(f"{sys.executable} laya/run_ft_laya.py --checkpoint {SM}/checkpoint --run-id {RUN_ID}-b1 --smoke --out {OUT}/eval-smoke", cwd=FT)
    print(open(f"{OUT}/eval-smoke/{RUN_ID}-b1-smoke/validation.jsonl").read()[:3000])
    print("B1 PASSED")

In [ ]:
# ---- b2: pilot. 1 epoch on all train rows, calibrate, score 50 validation rows; project b2 + b3 (recorded, not gated: FT-C-13 amended) ----
if "b2" in STAGES:
    PL = f"{OUT}/{RUN_ID}-b2"
    shutil.rmtree(PL, ignore_errors=True)
    sh(f"{sys.executable} laya/train_laya.py --data-dir data/laya --out {PL} --epochs 1 --no-sanity", cwd=FT)
    sh(f"{sys.executable} laya/run_ft_laya.py --checkpoint {PL}/checkpoint --run-id {RUN_ID}-b2 --split validation --limit 50 --out {OUT}/eval-pilot", cwd=FT)
    rep = json.load(open(f"{PL}/train_report.json")); e1 = rep["epochs"][1]
    per_epoch = e1["train_seconds"] + e1["save_seconds"] + e1["score_seconds"]
    total_h = (rep["total_wall_seconds"] + 3 * per_epoch + 300) / 3600     # pilot (1 epoch + loads + calibration) + 3 more epochs + a 5 min margin
    print(f"epoch {e1['train_seconds']/60:.1f} min train + {e1['score_seconds']/60:.1f} min development scoring + save {e1['save_seconds']:.0f}s | "
          f"peak GPU {rep['peak_gpu_mem_mb']} MB | projected b3 (4 epochs) about {total_h:.2f} h of a {SESSION_HOURS} h session")
    print("RECORD THIS PROJECTION in the run notes.")
    assert total_h < SESSION_HOURS, "BLOCKED: projection exceeds one Kaggle session: stop and ask the owner"
    print("B2 PASSED")

In [ ]:
# ---- b3: full run (4 epochs, per-epoch checkpoints, development route macro-F1 per epoch, calibration). Never retrained if finished ----
if "b3" in STAGES:
    if not os.path.exists(f"{RUN}/train_report.json"):
        assert not os.path.exists(RUN), f"{RUN} holds an unfinished run: delete it by hand or use a new run id (-b only with the owner's go-ahead)"
        rc = sh(f"{sys.executable} laya/train_laya.py --data-dir data/laya --out {RUN}", cwd=FT, check=False)   # exit 3 = sanity check failed (FT-C-12)
        assert rc == 0, f"BLOCKED: train_laya.py exited {rc} (3 = sanity check failed, 2 = skipped items / collapse): stop and report"
    else:
        print("run already trained: skipping training")
    rep = json.load(open(f"{RUN}/train_report.json"))
    for h in rep["epochs"]: print(h)
    print("selected epoch", rep["epoch_selected"], "| temperatures", rep["temperature"], "| sanity", rep["sanity"])

In [ ]:
# ---- restore: a fresh session has an empty /kaggle/working. Copy the checkpoint (and train_report.json) from the private input Dataset ----
if "b4_validation" in STAGES or "b4_test" in STAGES:
    if not os.path.exists(f"{RUN}/checkpoint/rl_agent_config.json"):
        cfgs = glob.glob("/kaggle/input/**/checkpoint/rl_agent_config.json", recursive=True) + glob.glob("/kaggle/input/**/rl_agent_config.json", recursive=True)
        cfgs = [c for c in cfgs if "laya-ft-bundle" not in c]
        assert cfgs, "no checkpoint in /kaggle/input: add the private checkpoint Dataset (Add Input) and re-run"
        ck_src = os.path.dirname(cfgs[0])
        os.makedirs(RUN, exist_ok=True)
        shutil.copytree(ck_src, f"{RUN}/checkpoint")
        for name in ("train_report.json", "epochs.jsonl"):
            for base in (os.path.dirname(ck_src), ck_src):
                if os.path.exists(f"{base}/{name}") and not os.path.exists(f"{RUN}/{name}"):
                    shutil.copy(f"{base}/{name}", f"{RUN}/{name}")
        print("restored checkpoint from", ck_src)
    assert os.path.exists(f"{RUN}/train_report.json"), "train_report.json missing next to the checkpoint: add it to the Dataset"
    import hashlib
    h = hashlib.sha256(open(f"{RUN}/checkpoint/model.safetensors", "rb").read()).hexdigest()
    exp = json.load(open(f"{RUN}/train_report.json"))["weights_sha256"]
    assert h == exp, f"checkpoint sha256 {h} != train_report {exp}"
    print("checkpoint sha256 verified", h[:16])


In [ ]:
# ---- b4: round-1 harness on the fine-tuned checkpoint (validation first; test once after FROZEN) ----
def eval_split(split):
    sh(f"{sys.executable} laya/run_ft_laya.py --checkpoint {RUN}/checkpoint --run-id {RUN_ID} --split {split} --out {OUT}"
       + (f" --kaggle-dataset {KAGGLE_DATASET}" if KAGGLE_DATASET else ""), cwd=FT)   # exit != 0 on a rejected log

if "b4_validation" in STAGES:
    eval_split("validation")
    m = json.load(open(f"{RUN}/metrics_validation.json"))
    print(f"validation route macro-F1 {m['route']['macro_f1']:.3f} (accuracy {m['route']['accuracy']:.3f}) vs laya-v2-a 0.395 | error rows {m['n_error_rows']}")
    from IPython.display import Markdown, display
    display(Markdown(open(f"{RUN}/report_validation.md").read()))
    print("Review the validation report, then set FROZEN = True for b4_test")

if "b4_test" in STAGES:
    assert FROZEN, "b4_test runs only after the configuration is frozen (FT-C-14): set FROZEN = True by hand"
    assert not os.path.exists(f"{RUN}/test.jsonl"), "test already run for this configuration: a failed run is re-run identically, never changed"
    eval_split("test")
    from IPython.display import Markdown, display
    display(Markdown(open(f"{RUN}/report_test.md").read()))

In [ ]:
# ---- checkpoint hash and listing (record in run.json; upload the checkpoint folder as a private Dataset) ----
if os.path.exists(f"{RUN}/checkpoint"):
    sh(f"cd {RUN}/checkpoint && ls -la && sha256sum model.safetensors")
sh(f"du -sh /kaggle/working; ls {OUT}")